Exploring the data on the excel sheets i"ve noticed

Data diagnosis:

There are NANs in two of the variables, probably, capture errors

There are negative weights, it is fisically impossible, i"m assuming those are capture errors to

There are three variables with categorical data, an encoding method is needed

In [1]:
import pandas as pd
import numpy as np
import itertools
import random
from datetime import datetime

import os
import json
import pickle

from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn.impute import KNNImputer

#from sklearn.linear_model import Lasso
#from sklearn.linear_model import LassoCV
#from sklearn.linear_model import LinearRegression
#from sklearn.metrics import mean_squared_error
#from sklearn.metrics import r2_score

import statsmodels.api as sm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.models import load_model

The data contains variables that are going to be ignored in the model due its nature, this ones are:

1. date
2. delivery_lat
3. delivery_lon
4. pickup
5. pickup_lat
6. pickup_lon

Noticing "equipment" has only three posible responses i have one-hot encoded it into three dummy variables where

D = Dry Van,
F = Flatbed,
R = Reefer.

Then, the pickup and delivery variables were target encoded, using the mean posted_rate associated to its entry.

Now, let"s start adjusting a neural network model.

In [2]:
#BENCH
tren = pd.read_csv("C:/Users/corazon/Desktop/Apply/train_corrected.csv")
tren.dtypes

load_id             str
pickup          float64
delivery        float64
distance        float64
D                 int64
F                 int64
R               float64
weight          float64
market_index    float64
quote_signal    float64
posted_rate     float64
dtype: object

In [3]:
x = tren.drop({"load_id","posted_rate"}, axis=1)
y = tren["posted_rate"]
x.corr()

,pickup,delivery,distance,D,F,R,weight,market_index,quote_signal
pickup,1.000000,-0.013023,-0.035663,0.000858,-0.003509,0.002150,-0.001915,-0.002173,-0.002395
delivery,-0.013023,1.000000,-0.029591,0.000735,-0.003070,0.001893,0.004557,0.001075,0.007570
distance,-0.035663,-0.029591,1.000000,0.002232,-0.002166,-0.000636,0.002544,-0.002181,-0.057244
D,0.000858,0.000735,0.002232,1.000000,-0.540088,-0.661929,-0.000189,0.001954,-0.058944
F,-0.003509,-0.003070,-0.002166,-0.540088,1.000000,-0.273345,-0.003279,-0.001677,0.027676
R,0.002150,0.001893,-0.000636,-0.661929,-0.273345,1.000000,0.003136,-0.000749,0.042736
weight,-0.001915,0.004557,0.002544,-0.000189,-0.003279,0.003136,1.000000,-0.001331,0.008159
market_index,-0.002173,0.001075,-0.002181,0.001954,-0.001677,-0.000749,-0.001331,1.000000,-0.067827
quote_signal,-0.002395,0.007570,-0.057244,-0.058944,0.027676,0.042736,0.008159,-0.067827,1.000000


In [4]:
x.isna().sum()

pickup            0
delivery          0
distance          0
D                 0
F                 0
R                 1
weight          300
market_index    374
quote_signal      0
dtype: int64

Now, for the NAs in weight and market_index i"m going to input the missing data using k-neighbours method.
First, reescalate the data using MinMaxScaler

In [5]:
escalera=preprocessing.MinMaxScaler()
escalados=escalera.fit_transform(x)
escalados = pd.DataFrame(escalados, columns=x.columns, index=x.index)

In [6]:
sixput=KNNImputer(n_neighbors=6, weights="distance")
x=sixput.fit_transform(escalados)

In [7]:
x_temp, x_test, y_temp, y_test = train_test_split(
    x, y, 
    test_size=0.2,
    random_state=67)

x_train, x_val, y_train, y_val = train_test_split(
    x_temp, y_temp, 
    test_size=0.125,
    random_state=67
)
def crear_modelo(
    n_entradas,
    n_capas_ocultas=2,
    n_neurons=64,
    dropout_rate=0.2,
    lr=0.001, activation="relu"
):

    model = models.Sequential()
    model.add(layers.Input(shape=(n_entradas,)))
    for i in range(n_capas_ocultas):
        model.add(layers.Dense(n_neurons, activation=activation))
        model.add(layers.Dropout(dropout_rate))
        model.add(layers.BatchNormalization())
    model.add(layers.Dense(1, activation="linear"))
    optimizer = keras.optimizers.Adam(learning_rate=lr)
    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"])
    
    return model

In [8]:
N_COMBINACIONES = 15
EPOCHS_MAXIMOS = 100
PATIENCE_EARLY_STOPPING = 10

hiperparametros = {
    "n_capas_ocultas": [1, 2, 3, 4],
    "n_neurons": [16, 32, 64, 128],
    "dropout_rate": [0.1, 0.2, 0.3, 0.4],
    "lr": [0.01, 0.001, 0.0005, 0.0001],
    "batch_size": [8, 16, 32, 64],
    "activation": ["relu", "tanh", "sigmoid"]
}

In [9]:
RESULTADOS_DIR = "resultados_entrenamiento"
os.makedirs(RESULTADOS_DIR, exist_ok=True)

def guardar_modelo(model, nombre_archivo):
    model.save(f"{nombre_archivo}.keras")


def guardar_resultados(resultados, nombre_archivo="resultados_totales"):
    resultados_serializables = []
    for r in resultados:
        r_serializable = {
            "params": r["params"],
            "val_loss": float(r["val_loss"]),
            "val_mae": float(r["val_mae"]),
            "history_epochs": len(r["history"].history["loss"]) if r["history"] else 0
        }
        resultados_serializables.append(r_serializable)
    
    with open(f"{RESULTADOS_DIR}/{nombre_archivo}.json", "w") as f:
        json.dump(resultados_serializables, f, indent=2)
    
    with open(f"{RESULTADOS_DIR}/{nombre_archivo}_completo.pkl", "wb") as f:
        pickle.dump(resultados, f)
    
    print(f"   Results saved: {RESULTADOS_DIR}/{nombre_archivo}")

def crear_modelo(
    n_entradas,
    n_capas_ocultas=2,
    n_neurons=64,
    dropout_rate=0.2,
    lr=0.001,
    activation="relu"
):
    model = models.Sequential()
    model.add(layers.Input(shape=(n_entradas,)))
    for i in range(n_capas_ocultas):
        model.add(layers.Dense(n_neurons, activation=activation))
        model.add(layers.Dropout(dropout_rate))
        model.add(layers.BatchNormalization())
    model.add(layers.Dense(1, activation="linear"))

    optimizer = keras.optimizers.Adam(learning_rate=lr)
    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )
    
    return model

def evaluar_hiperparametros(params, x_train, x_val, y_train, y_val, combinacion_id):
    combo_dir = f"{RESULTADOS_DIR}/combinacion_{combinacion_id}"
    os.makedirs(combo_dir, exist_ok=True)
    
    model = crear_modelo(
        n_entradas=x_train.shape[1],
        n_capas_ocultas=params["n_capas_ocultas"],
        n_neurons=params["n_neurons"],
        dropout_rate=params["dropout_rate"],
        lr=params["lr"],
        activation=params["activation"]
    )

    early_stopping = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE_EARLY_STOPPING,
        restore_best_weights=True,
        verbose=1)
    
    checkpoint = callbacks.ModelCheckpoint(
        filepath=f"{combo_dir}/mejor_modelo.keras",
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    )
    
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=EPOCHS_MAXIMOS,
        batch_size=params["batch_size"],
        callbacks=[early_stopping, checkpoint],
        verbose=1)
    model.load_weights(f"{combo_dir}/mejor_modelo.keras")
    val_loss, val_mae = model.evaluate(x_val, y_val, verbose=0)
    with open(f"{combo_dir}/history.json", "w") as f:
        history_dict = {
            "loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
            "mae": history.history["mae"],
            "val_mae": history.history["val_mae"],
            "epochs": len(history.history["loss"])
        }
        json.dump(history_dict, f, indent=2)

    with open(f"{combo_dir}/params.json", "w") as f:
        json.dump(params, f, indent=2)
    
    metricas = {
        "val_loss": float(val_loss),
        "val_mae": float(val_mae),
        "epochs_entrenados": len(history.history["loss"]),
        "timestamp": datetime.now().isoformat()
    }
    with open(f"{combo_dir}/metricas.json", "w") as f:
        json.dump(metricas, f, indent=2)
    
    return {
        "model": model,
        "val_loss": val_loss,
        "val_mae": val_mae,
        "history": history,
        "params": params,
        "combinacion_id": combinacion_id,
        "directorio": combo_dir
    }

MEJORES_MODELOS = []

def continuar_entrenamiento(mejor_modelo_info, x_train, x_val, y_train, y_val, epochs_extra=50):
    modelo = mejor_modelo_info["model"]
    params = mejor_modelo_info["params"]
    combo_dir = f"{RESULTADOS_DIR}/mejor_modelo_final"
    os.makedirs(combo_dir, exist_ok=True)
    
    early_stopping = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE_EARLY_STOPPING,
        restore_best_weights=True,
        verbose=1
    )
    
    # Aquí checas el punto
    checkpoint = callbacks.ModelCheckpoint(
        filepath=f"{combo_dir}/mejor_modelo_final.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
    
    # Controlar exigencia por si no avanza
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=10,
        min_lr=1e-6,
        verbose=1
    )
    
    historia_extra = modelo.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs_extra,
        batch_size=params["batch_size"],
        callbacks=[early_stopping, checkpoint, reduce_lr],
        verbose=1
    )
    
    # Modelo final
    modelo.save(f"{combo_dir}/modelo_final.keras")
    modelo.save(f"{combo_dir}/modelo_final.h5")

    with open(f"{combo_dir}/historia_extra.json", "w") as f:
        json.dump(historia_extra.history, f, indent=2)
    
    print(f"   Modelo final guardado en: {combo_dir}")
    return modelo, historia_extra

mejor_resultado = None
mejor_loss = float("inf")
resultados_totales = []

timestamp_inicio = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = f"{RESULTADOS_DIR}/entrenamiento_{timestamp_inicio}.log"

with open(log_file, "w") as f:
    f.write(f"Inicio del entrenamiento: {datetime.now().isoformat()}\n")

for i in range(N_COMBINACIONES):
    print(f"\n{'='*60}")
    print(f"COMBINACIÓN {i+1}/{N_COMBINACIONES}")
    
    params = {
        "n_capas_ocultas": random.choice(hiperparametros["n_capas_ocultas"]),
        "n_neurons": random.choice(hiperparametros["n_neurons"]),
        "dropout_rate": random.choice(hiperparametros["dropout_rate"]),
        "lr": random.choice(hiperparametros["lr"]),
        "batch_size": random.choice(hiperparametros["batch_size"]),
        "activation": random.choice(hiperparametros["activation"])
    }
    
    for key, value in params.items():
        print(f"   {key}: {value}")
    
    resultado = evaluar_hiperparametros(
        params, 
        x_train,
        x_val,
        y_train, 
        y_val,
        combinacion_id=i+1
    )
    
    resultados_totales.append(resultado)

    print(f"\n   Resultados:")
    print(f"   Loss (MSE): {resultado['val_loss']:.4f}")
    print(f"   MAE:        {resultado['val_mae']:.4f}")
    print(f"   Épocas:     {len(resultado['history'].history['loss'])}")
    
    if resultado["val_loss"] < mejor_loss:
        mejor_loss = resultado["val_loss"]
        mejor_resultado = resultado
        
        mejor_dir = f"{RESULTADOS_DIR}/mejor_hasta_ahora"
        os.makedirs(mejor_dir, exist_ok=True)
        resultado["model"].save(f"{mejor_dir}/mejor_modelo_comb_{i+1}.keras")

    if (i + 1) % 5 == 0:
        guardar_resultados(resultados_totales, f"resultados_parciales_{i+1}")
    
    with open(log_file, "a") as f:
        f.write(f"Combinación {i+1}: loss={resultado['val_loss']:.4f}, mae={resultado['val_mae']:.4f}\n")
        if resultado["val_loss"] < mejor_loss:
            f.write("   *** NUEVO MEJOR MODELO ***\n")

guardar_resultados(resultados_totales, "resultados_finales")

print(f"DONE")



COMBINACIÓN 1/15
   n_capas_ocultas: 1
   n_neurons: 64
   dropout_rate: 0.2
   lr: 0.0005
   batch_size: 64
   activation: tanh
Epoch 1/100
476/525 ━━━━━━━━━━━━━━━━━━━━ 0s 872us/step - loss: 7880900.0000 - mae: 2374.1782
Epoch 1: val_loss improved from None to 7562972.00000, saving model to resultados_entrenamiento/combinacion_1/mejor_modelo.keras

Epoch 1: finished saving model to resultados_entrenamiento/combinacion_1/mejor_modelo.keras
525/525 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 7878449.0000 - mae: 2373.6807 - val_loss: 7562972.0000 - val_mae: 2357.7913
Epoch 2/100
495/525 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 7669077.0000 - mae: 2354.4407
Epoch 2: val_loss improved from 7562972.00000 to 7263709.00000, saving model to resultados_entrenamiento/combinacion_1/mejor_modelo.keras

Epoch 2: finished saving model to resultados_entrenamiento/combinacion_1/mejor_modelo.keras
525/525 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 7678021.5000 - mae: 2353.5627 - val_loss: 7263709.0000 - va

Now, the neural network model is set and trained, let"s predict.
But first, let"s transform and encode the data in validation.csv

Now, validation has NaNs, the inputation will be the same method with k=6 and the codificaction will be the same...

In [13]:
fates = pd.read_csv("C:/Users/corazon/Desktop/Apply/validationcorr.csv")
fate = fates.drop(columns=["load_id"])
ladder = preprocessing.MinMaxScaler()
fate_scaled = ladder.fit_transform(fate)

fate = pd.DataFrame(fate_scaled, columns=fates.columns.drop("load_id"), index=fates.index)
sixput=KNNImputer(n_neighbors=6, weights="distance")
fate=sixput.fit_transform(fate)
modelo = mejor_resultado["model"]
modelo.save("modelo.keras")
preds=modelo.predict(fate)

375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 546us/step


Now, merge the predicts with the validation template

In [14]:
preds=modelo.predict(fate)
template = pd.read_csv("C:/Users/corazon/Desktop/Apply/validation-predictions-template.csv")
template["predicted_rate"] = preds
template.to_csv("validation_predictions.csv", index=False)

375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 556us/step


Diciembre

As only five of the seven variables have a value on, i'll input market_index and quote_signal with a linear regression on those values, we dont know whats really happening on the black box so i wont force an distribution for the dataset.

In [119]:
og = pd.read_csv("C:/Users/corazon/Desktop/Apply/december-chart-inputs.csv")
xmas = pd.read_csv("C:/Users/corazon/Desktop/Apply/december-chart-inputs-corr.csv")
tren = pd.read_csv("C:/Users/corazon/Desktop/Apply/train_corrected.csv")

x = tren.drop({"load_id","posted_rate"}, axis=1)
y = tren["posted_rate"]

escalera = preprocessing.MinMaxScaler()
escalados = escalera.fit_transform(x)
escalados = pd.DataFrame(escalados, columns=x.columns, index=x.index)

sixput = KNNImputer(n_neighbors=6, weights="distance")
x_imputado_escalado = sixput.fit_transform(escalados)
x_imputado_escalado = pd.DataFrame(x_imputado_escalado, columns=x.columns, index=x.index)

x_imputado = pd.DataFrame(
    escalera.inverse_transform(x_imputado_escalado),
    columns=x.columns,
    index=x.index
)


features = ['pickup', 'delivery', 'distance', 'D', 'F', 'R', 'weight']
X = x_imputado[features]
y_market = x_imputado['market_index']
y_quote = x_imputado['quote_signal']

# Modelos
X_sm = sm.add_constant(X)
modelo_market = sm.OLS(y_market, X_sm).fit()
modelo_quote = sm.OLS(y_quote, X_sm).fit()

print(modelo_market.summary())
print(modelo_quote.summary())

                            OLS Regression Results                            
Dep. Variable:           market_index   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.1413
Date:                Tue, 04 Aug 2026   Prob (F-statistic):              0.991
Time:                        20:21:19   Log-Likelihood:                 17620.
No. Observations:               48000   AIC:                        -3.523e+04
Df Residuals:                   47993   BIC:                        -3.516e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.8247      0.046     18.098      0.0

It is clear at first glance these are farm from being good models, but this ones are the ones i'm using

In [120]:
#latabla = pd.DataFrame({
#    'pickup': [2364.243526],
#    'delivery': [2343.268614],
#   'distance': [360],
#    'D': [1],
#    'F': [0],
#    'R': [0],
#    'weight': [32000]
#})

#latabla_sm = latabla.copy()
#latabla_sm['const'] = 1 
#latabla_sm = latabla_sm[X_sm.columns]

#market_pred = modelo_market.predict(latabla_sm)[0]
#quote_pred = modelo_quote.predict(latabla_sm)[0]

In [121]:
xpreder = xmas[features].copy()
xpreder['const'] = 1
xpreder = xpreder[X_sm.columns]
xmas['market_index'] = modelo_market.predict(xpreder)
xmas['quote_signal'] = modelo_quote.predict(xpreder)

In [123]:
xmaspreds=modelo.predict(xmas)
og["predicted_rate"] = xmaspreds
og
og.to_csv("december-chart-outputs.csv", index=False)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
